<a href="https://colab.research.google.com/github/Sasidhar015/Data-engineering-training-2026/blob/main/26_10_2016PySpark/New_Operations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
%pip -q install pyspark

In [3]:

from google.colab import files, output


output.enable_custom_widget_manager()

import pandas as pd
import ipywidgets as widgets
import json
import os
from IPython.display import display


uploaded = files.upload()


datasets = {}

for file_name in uploaded.keys():

    print(f"\nReading file: {file_name}")


    if file_name.lower().endswith(".csv"):
        df = pd.read_csv(file_name)
        datasets[file_name] = df

        print("File type: CSV")
        print("Number of records:", len(df))
        print("Number of columns:", len(df.columns))

        display(df.head())


    elif file_name.lower().endswith(".json"):
        with open(file_name, "r", encoding="utf-8") as file:
            data = json.load(file)

        datasets[file_name] = data

        print("File type: JSON")
        print("JSON file loaded successfully!")

        if isinstance(data, (dict, list)):
            display(data)
        else:
            print(data)

    else:
        print("Unsupported file format. Skipping this file.")




Saving manufacturing_production_extra_12k.csv to manufacturing_production_extra_12k.csv

Reading file: manufacturing_production_extra_12k.csv
File type: CSV
Number of records: 12500
Number of columns: 18


,production_id,production_date,plant_id,machine_id,operator_id,shift,product_code,product_family,units_planned,units_produced,defective_units,downtime_minutes,energy_kwh,temperature_c,vibration_mm_s,quality_status,maintenance_flag,supervisor
0,PR0120001,2026-06-01,PL05,MC024,OP0191,Evening,PMP-210,Pumps,293,292,10,44.0,120.42,70.59,2.21,Pass,Yes,Anita
1,PR0120002,2026-09-22,PL01,MC001,OP0461,Night,MTR-100,Motors,1009,897,25,59.0,317.23,81.74,3.03,Pass,No,Anita
2,PR0120003,2026-05-24,PL03,MC012,OP0451,Evening,MTR-100,Motors,960,983,34,0.0,350.82,67.29,2.01,Pass,No,Deepa
3,PR0120004,2026-09-12,PL04,MC016,OP0237,Night,CTL-130,Controllers,310,320,15,14.0,142.34,59.70,3.48,Pass,No,Anita
4,PR0120005,2026-02-04,PL04,MC018,OP0072,Morning,VLV-220,Valves,957,979,22,14.0,296.70,62.52,2.33,Pass,No,Harini


In [4]:
# Function to create dropdown options
def create_options(column, all_label):
    values = sorted(df[column].dropna().astype(str).unique())

    return [(all_label, "__ALL__")] + [
        (value, value) for value in values
    ]


# 1. Plant dropdown
plant_dropdown = widgets.Dropdown(
    options=create_options("plant_id", "All Plants"),
    description="Plant:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="260px")
)


# 2. Product family dropdown
product_dropdown = widgets.Dropdown(
    options=create_options("product_family", "All Products"),
    description="Product Family:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="300px")
)


# 3. Quality status dropdown
quality_dropdown = widgets.Dropdown(
    options=create_options("quality_status", "All Statuses"),
    description="Quality:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="260px")
)


# 4. Slider for units produced
units = pd.to_numeric(df["units_produced"], errors="coerce").dropna()

units_slider = widgets.IntRangeSlider(
    value=(int(units.min()), int(units.max())),
    min=int(units.min()),
    max=int(units.max()),
    step=1,
    description="Units Produced:",
    continuous_update=False,
    style={"description_width": "initial"},
    layout=widgets.Layout(width="90%")
)

In [6]:
from IPython.display import display
from google.colab import output

# Enable interactive widgets in Google Colab
output.enable_custom_widget_manager()

# Display all four widgets
display(widgets.VBox([
    plant_dropdown,
    product_dropdown,
    quality_dropdown,
    units_slider
]))


In [7]:

df = df.sort_values(by="plant_id", ascending=True)


def create_options(column, all_label):
    values = sorted(df[column].dropna().astype(str).unique())

    return [(all_label, "__ALL__")] + [(value, value) for value in values]

plant_dropdown = widgets.Dropdown(
    options=create_options("plant_id", "All Plants"),
    description="Plant:")


display(plant_dropdown)

Dropdown(description='Plant:', options=(('All Plants', '__ALL__'), ('PL01', 'PL01'), ('PL02', 'PL02'), ('PL03'…

In [9]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, avg, desc

spark = SparkSession.builder \
    .appName("LazyExecutionDemo") \
    .getOrCreate()

data = [
    (1, "Hyderabad", "Laptop", 2, 55000),
    (2, "Bangalore", "Mobile", 5, 20000),
    (3, "Chennai", "Tablet", 3, 15000),
    (4, "Hyderabad", "Mobile", 4, 22000),
    (5, "Mumbai", "Laptop", 1, 60000),
    (6, "Bangalore", "Tablet", 6, 12000),
    (7, "Chennai", "Laptop", 2, 58000),
    (8, "Mumbai", "Mobile", 8, 18000),
    (9, "Hyderabad", "Tablet", 5, 14000),
    (10, "Bangalore", "Laptop", 3, 52000),
    (11, "Mumbai", "Tablet", 7, 13000),
    (12, "Chennai", "Mobile", 6, 21000)]

columns = ["sale_id", "city", "product", "quantity", "price"]

df = spark.createDataFrame(data, columns)

df.show()

+-------+---------+-------+--------+-----+
|sale_id|     city|product|quantity|price|
+-------+---------+-------+--------+-----+
|      1|Hyderabad| Laptop|       2|55000|
|      2|Bangalore| Mobile|       5|20000|
|      3|  Chennai| Tablet|       3|15000|
|      4|Hyderabad| Mobile|       4|22000|
|      5|   Mumbai| Laptop|       1|60000|
|      6|Bangalore| Tablet|       6|12000|
|      7|  Chennai| Laptop|       2|58000|
|      8|   Mumbai| Mobile|       8|18000|
|      9|Hyderabad| Tablet|       5|14000|
|     10|Bangalore| Laptop|       3|52000|
|     11|   Mumbai| Tablet|       7|13000|
|     12|  Chennai| Mobile|       6|21000|
+-------+---------+-------+--------+-----+



In [10]:

# Transformation 1
filtered_df = df.filter(col("quantity") > 3)

print("Transformation created")

# Transformation 2
selected_df = filtered_df.select(
    "city", "product", "quantity"
)

print("Another transformation created")

# Action
selected_df.show()



Transformation created
Another transformation created
+---------+-------+--------+
|     city|product|quantity|
+---------+-------+--------+
|Bangalore| Mobile|       5|
|Hyderabad| Mobile|       4|
|Bangalore| Tablet|       6|
|   Mumbai| Mobile|       8|
|Hyderabad| Tablet|       5|
|   Mumbai| Tablet|       7|
|  Chennai| Mobile|       6|
+---------+-------+--------+



In [11]:

from pyspark.sql.functions import col, sum, desc

step1 = df.filter(col("quantity") >= 3)

# Transformation 2: Calculate revenue
step2 = step1.withColumn(
    "revenue",
    col("quantity") * col("price"))

step3 = step2.groupBy("city").agg(
    sum("revenue").alias("total_revenue"))

step4 = step3.orderBy(desc("total_revenue"))

print("All transformations defined")
step4.show()



All transformations defined
+---------+-------------+
|     city|total_revenue|
+---------+-------------+
|Bangalore|       328000|
|   Mumbai|       235000|
|  Chennai|       171000|
|Hyderabad|       158000|
+---------+-------------+



In [13]:
result = df.filter(col("price") > 20000) \
           .select("city", "product", "price") \
           .orderBy(desc("price"))


result.explain()



== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [price#4L DESC NULLS LAST], true, 0
   +- Exchange rangepartitioning(price#4L DESC NULLS LAST, 200), ENSURE_REQUIREMENTS, [plan_id=123]
      +- Project [city#1, product#2, price#4L]
         +- Filter (isnotnull(price#4L) AND (price#4L > 20000))
            +- Scan ExistingRDD[sale_id#0L,city#1,product#2,quantity#3L,price#4L]




In [14]:
result.show()

+---------+-------+-----+
|     city|product|price|
+---------+-------+-----+
|   Mumbai| Laptop|60000|
|  Chennai| Laptop|58000|
|Hyderabad| Laptop|55000|
|Bangalore| Laptop|52000|
|Hyderabad| Mobile|22000|
|  Chennai| Mobile|21000|
+---------+-------+-----+



In [16]:
high_value_sales = df.filter(col("price") > 20000)


high_value_sales.show()


print("Total records:", high_value_sales.count())

records = high_value_sales.collect()
print(records)


+-------+---------+-------+--------+-----+
|sale_id|     city|product|quantity|price|
+-------+---------+-------+--------+-----+
|      1|Hyderabad| Laptop|       2|55000|
|      4|Hyderabad| Mobile|       4|22000|
|      5|   Mumbai| Laptop|       1|60000|
|      7|  Chennai| Laptop|       2|58000|
|     10|Bangalore| Laptop|       3|52000|
|     12|  Chennai| Mobile|       6|21000|
+-------+---------+-------+--------+-----+

Total records: 6
[Row(sale_id=1, city='Hyderabad', product='Laptop', quantity=2, price=55000), Row(sale_id=4, city='Hyderabad', product='Mobile', quantity=4, price=22000), Row(sale_id=5, city='Mumbai', product='Laptop', quantity=1, price=60000), Row(sale_id=7, city='Chennai', product='Laptop', quantity=2, price=58000), Row(sale_id=10, city='Bangalore', product='Laptop', quantity=3, price=52000), Row(sale_id=12, city='Chennai', product='Mobile', quantity=6, price=21000)]
